# 01_syntgen: Synthetic patient generator
Run cells from top to bottom. After **Restart**, use **Run All**.

## Setup

In [1]:
from datetime import date
import numpy as np, pandas as pd
import ipytest
ipytest.autoconfig()

CFG = {
    "seed": 20261101,
    "n_patients": 5000,
    "window": {"start_year": 2021, "end_year": 2026},
    "visits": {"max": 8},
    "visit_gap_months": {"Low": 12, "Moderate": 4, "High": 1.5},
    "family_multiplier": 2.0,
    "relatives": {"confirmed_share": 0.70, "unknown_onset_share": 0.20},
    "paths": {"healthy": 0.50, "progressor": 0.25, "already_ill": 0.15, "improver": 0.10},
    "fhrs": {
        "R": {1: 0.50, 2: 0.25, 3: 0.125},
        "S": {"confirmed": 1.0, "not_sure": 0.5},
        "cluster_bonus": 1.0,
        "penalties": {
            "CVD": {"secondhand_smoke": 0.15, "high_salt_diet": 0.15},
            "T2D": {"processed_diet": 0.15, "low_home_activity": 0.15},
            "CRD": {"secondhand_smoke": 0.15, "indoor_mold": 0.15, "coal_heating": 0.15},
            "NEURO": {},
        },
    },
    "drift_per_year": {"hba1c": {"healthy": 0.02, "progressor": 0.20, "improver": -0.10}},
    "missing": {"optional_lab": 0.70, "hospital_only": 0.93},
    "label_window_years": 5,
    "gbd_file": "data/gbd_2023_ph_rates.csv",
    "deaths_file": "data/psa_deaths_2024.csv",
    "pop_file": "data/psa_population_projection.csv",
    "death_year": "2024",
    "lifetime_target": {
        "Major depressive disorder": {"Female": 0.148, "Male": 0.074},
        "Anxiety disorders": {"Female": 0.016, "Male": 0.016},
        "Asthma": {"Female": 0.10, "Male": 0.10},
    },
    "checks": {
        "hypertension": (0.17, 0.23),
        "current_smoker": 0.204,
        "current_depression": 0.033,
    },
}
RNG = np.random.default_rng(CFG["seed"])
print("Setup OK")

Setup OK


Part 0: Prepare disease rates

In [2]:
# Cell 2 - CODE
AGE_GROUPS = [f"{a}-{a + 4} years" for a in range(15, 95, 5)] + ["95+ years"]

def load_rates(path):
    g = pd.read_csv(path)
    g = g[g["age_name"].isin(AGE_GROUPS) & (g["metric_name"] == "Rate")]
    wide = g.pivot_table(index=["cause_name", "sex_name", "age_name"], 
                         columns="measure_name", values="val").reset_index()
    wide["age_lo"] = wide["age_name"].str.extract(r"^(\d+)").astype(int)
    return pd.DataFrame({
        "disease": wide["cause_name"], "sex": wide["sex_name"], "age_lo": wide["age_lo"],
        "incidence": wide["Incidence"] / 100_000,
        "prevalence": wide["Prevalence"] / 100_000,
    })

def yearly_rate(rates, disease, age, sex, factor=1.0):
    if age < 15: return 0.0
    lo = min(95, age - age % 5)
    row = rates[(rates["disease"] == disease) & (rates["sex"] == sex) & (rates["age_lo"] == lo)]
    return float(row["incidence"].iloc[0]) * factor

def lifetime_risk(rates, disease, sex, factor=1.0, start=15, end=80):
    free = 1.0
    for age in range(start, end):
        free *= 1 - min(yearly_rate(rates, disease, age, sex, factor), 0.99)
    return 1 - free

def calibrate_factor(rates, disease, sex, target, start=15, end=80):
    lo, hi = 0.0, 1.0
    for _ in range(60):
        mid = (lo + hi) / 2
        if lifetime_risk(rates, disease, sex, mid, start, end) < target:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2

In [3]:
%%ipytest -q
# Cell 1 - TEST
import pytest

RAW = pd.read_csv(CFG["gbd_file"])
R = load_rates(CFG["gbd_file"])

def test_keeps_only_the_17_five_year_age_groups():
    assert sorted(R["age_lo"].unique()) == list(range(15, 95, 5)) + [95]

def test_has_7_diseases_2_sexes():
    assert R["disease"].nunique() == 7 and set(R["sex"]) == {"Male", "Female"}
    assert len(R) == 7 * 17 * 2

def test_rate_is_chance_per_person_per_year():
    raw = RAW.query("measure_name == 'Incidence' and cause_name == 'Diabetes mellitus type 2' "
                    "and sex_name == 'Male' and age_name == '50-54 years'")["val"].item()
    assert yearly_rate(R, "Diabetes mellitus type 2", 52, "Male") == pytest.approx(raw / 100_000)

def test_rate_is_zero_under_15():
    assert yearly_rate(R, "Stroke", 10, "Female") == 0.0

def test_age_100_uses_95_plus_group():
    raw = RAW.query("measure_name == 'Incidence' and cause_name == 'Stroke' "
                    "and sex_name == 'Female' and age_name == '95+ years'")["val"].item()
    assert yearly_rate(R, "Stroke", 100, "Female") == pytest.approx(raw / 100_000)

def test_factor_scales_the_rate():
    base = yearly_rate(R, "Asthma", 40, "Male")
    assert yearly_rate(R, "Asthma", 40, "Male", factor=0.25) == pytest.approx(base * 0.25)

def test_raw_depression_rates_give_unrealistic_lifetime_risk():
    assert lifetime_risk(R, "Major depressive disorder", "Female") > 0.80

def test_calibration_hits_the_target_lifetime_risk():
    k = calibrate_factor(R, "Major depressive disorder", "Female", target=0.15)
    assert 0 < k < 1
    assert lifetime_risk(R, "Major depressive disorder", "Female", factor=k) == pytest.approx(0.15, abs=0.001)

In [4]:
# Cell 3 - RUN
APP_NAME = {"Major depressive disorder": "depression", "Anxiety disorders": "GAD",
            "Asthma": "asthma", "Chronic obstructive pulmonary disease": "COPD",
            "Diabetes mellitus type 2": "T2D"}

RATES = load_rates(CFG["gbd_file"])
RATES["disease"] = RATES["disease"].replace(APP_NAME)

# CVD = ischemic heart disease OR stroke (chance of at least one)
ihd = RATES[RATES["disease"] == "Ischemic heart disease"].set_index(["sex", "age_lo"])
stk = RATES[RATES["disease"] == "Stroke"].set_index(["sex", "age_lo"])
cvd = pd.DataFrame({"incidence": 1 - (1 - ihd["incidence"]) * (1 - stk["incidence"]),
                    "prevalence": ihd["prevalence"] + stk["prevalence"]}).reset_index()
cvd["disease"] = "CVD"
RATES = pd.concat([RATES, cvd[RATES.columns]], ignore_index=True)

TARGET = {APP_NAME.get(d, d): t for d, t in CFG["lifetime_target"].items()}
FACTOR = {(d, s): calibrate_factor(RATES, d, s, t)
          for d, by_sex in TARGET.items() for s, t in by_sex.items()}

# fast lookup: a dict is ~1000x faster than filtering the DataFrame each call
LOOKUP = {(d, s, a): i for d, s, a, i in RATES[["disease", "sex", "age_lo", "incidence"]].itertuples(index=False)}

SEX = {"M": "Male", "F": "Female", "Male": "Male", "Female": "Female"}

def rate_fn(disease, age, sex):
    sex = SEX[sex]
    if age < 15:
        return 0.0
    return LOOKUP[(disease, sex, min(95, age - age % 5))] * FACTOR.get((disease, sex), 1.0)

print("Factors:", FACTOR)
print(sorted(RATES["disease"].unique()))

Factors: {('depression', 'Female'): 0.07974342611145155, ('depression', 'Male'): 0.05495744013565233, ('GAD', 'Female'): 0.05582501169309552, ('GAD', 'Male'): 0.08667907839000985, ('asthma', 'Female'): 0.22618939865371007, ('asthma', 'Male'): 0.22759915460694619}
['COPD', 'CVD', 'GAD', 'Ischemic heart disease', 'Stroke', 'T2D', 'asthma', 'depression']


In [5]:
%%ipytest -q
# Cell 4 - TEST for the RUN cell (app disease names, CVD, sex codes)
import pytest

def test_rate_fn_knows_every_app_disease_name():
    for d in ["CVD", "T2D", "asthma", "COPD", "depression", "GAD"]:
        for s in ("Male", "Female"):
            assert rate_fn(d, 50, s) > 0

def test_cvd_rate_combines_ischemic_heart_disease_and_stroke():
    ihd = yearly_rate(RATES, "Ischemic heart disease", 60, "Male")
    stroke = yearly_rate(RATES, "Stroke", 60, "Male")
    assert rate_fn("CVD", 60, "Male") == pytest.approx(1 - (1 - ihd) * (1 - stroke))

def test_asthma_calibration_factor_is_used():
    raw = yearly_rate(RATES, "asthma", 40, "Female")
    assert rate_fn("asthma", 40, "Female") == pytest.approx(raw * FACTOR[("asthma", "Female")])

def test_rate_fn_is_zero_under_15():
    assert rate_fn("T2D", 10, "Male") == 0.0

def test_rate_fn_accepts_short_sex_codes_used_by_fhrs():
    assert rate_fn("T2D", 50, "F") == rate_fn("T2D", 50, "Female")

Part 0b: Prepare death rates

In [6]:
# Cell 2 - CODE
def load_death_rates(deaths_path, pop_path, year="2024"):
    d = pd.read_csv(deaths_path, skiprows=1)
    d["Age Group"] = d["Age Group"].replace({"Under 1": "0-4", "1-4": "0-4", "85 and over": "85+"})
    d = d[~d["Age Group"].isin(["Total", "Not Stated"])]
    d = d.groupby("Age Group", sort=False)[["Male", "Female"]].sum()
    
    p = pd.read_csv(pop_path, skiprows=1)
    rows = []
    for sex in ("Male", "Female"):
        pop = p[p["Sex"] == sex].set_index("Age Group")[year] * 1000
        for ag in d.index:
            lo = 85 if ag == "85+" else int(ag.split("-")[0])
            rows.append({"sex": sex, "age_lo": lo, "rate": d.loc[ag, sex] / pop[ag]})
    return pd.DataFrame(rows)

def death_rate(dr, age, sex):
    sex = {"M": "Male", "F": "Female"}.get(sex, sex)
    lo = min(85, age - age % 5)
    return float(dr[(dr["sex"] == sex) & (dr["age_lo"] == lo)]["rate"].iloc[0])

def life_expectancy(dr, sex):
    m = dr[dr["sex"] == sex].sort_values("age_lo")["rate"].tolist()
    alive, years = 1.0, 0.0
    for mx in m[:-1]:
        q = 5 * mx / (1 + 2.5 * mx)
        years += 5 * alive * (1 - q / 2)
        alive *= 1 - q
    return years + alive / m[-1]

def draw_death_year(birth_year, sex, rng, dr, end_year=2026):
    for year in range(birth_year, end_year + 1):
        if rng.random() < death_rate(dr, year - birth_year, sex):
            return year
    return None   # still alive

In [7]:
%%ipytest -q
# Cell 1 - TEST
import pytest

DR = load_death_rates(CFG["deaths_file"], CFG["pop_file"], CFG["death_year"])

def test_population_in_thousands_is_converted():
    assert death_rate(DR, 62, "Male") == pytest.approx(42489 / 1_807_000)

def test_under_1_and_1_to_4_are_merged_into_0_to_4():
    assert death_rate(DR, 2, "Male") == pytest.approx((11714 + 3320) / 4_606_000)

def test_age_90_uses_85_plus_group():
    assert death_rate(DR, 90, "Female") == pytest.approx(53786 / 284_000)

def test_rates_rise_with_age_after_30():
    rates = [death_rate(DR, a, "Female") for a in range(30, 90, 5)]
    assert rates == sorted(rates)

def test_life_expectancy_is_believable():
    assert 65 < life_expectancy(DR, "Male") < 75
    assert 70 < life_expectancy(DR, "Female") < 82
    assert life_expectancy(DR, "Female") > life_expectancy(DR, "Male")

def test_death_year_is_after_birth_and_old_people_are_dead():
    rng = np.random.default_rng(3)
    years = [draw_death_year(1900, "Female", rng, DR) for _ in range(200)]
    assert all(y is not None and 1900 <= y <= 2026 for y in years)

def test_young_person_is_usually_alive():
    rng = np.random.default_rng(3)
    alive = [draw_death_year(2000, "Female", rng, DR) is None for _ in range(500)]
    assert np.mean(alive) > 0.9

def test_death_draw_accepts_short_sex_codes():
    assert draw_death_year(1900, "M", np.random.default_rng(1), DR) is not None

In [8]:
# Cell 3 - RUN
DR = load_death_rates(CFG["deaths_file"], CFG["pop_file"], CFG["death_year"])
print("Male Life Expectancy:", life_expectancy(DR, "Male"))
print("Female Life Expectancy:", life_expectancy(DR, "Female"))
print(DR.head())

Male Life Expectancy: 68.93999021366251
Female Life Expectancy: 75.28947913987874
    sex  age_lo      rate
0  Male       0  0.003264
1  Male       5  0.000423
2  Male      10  0.000482
3  Male      15  0.000974
4  Male      20  0.001440


Part 0c: Family size

In [9]:
# Cell 2 - CODE
TFR = {1993: 4.1, 1998: 3.7, 2003: 3.5, 2008: 3.3, 2013: 3.0, 2017: 2.7, 2022: 1.9}

def mean_siblings(birth_year):
    tfr = np.interp(birth_year, list(TFR), list(TFR.values()))
    return float(tfr) - 1

In [10]:
%%ipytest -q
# Cell 1 - TEST
import pytest

def test_mean_siblings_uses_ndhs_tfr_minus_one():
    assert mean_siblings(1993) == pytest.approx(3.1)

def test_mean_siblings_interpolates_between_surveys():
    assert mean_siblings(2010) == pytest.approx(3.3 + (3.0 - 3.3) * 2 / 5 - 1)

def test_births_before_1993_use_the_1993_value():
    assert mean_siblings(1965) == pytest.approx(3.1)

def test_sourced_targets_can_be_reached():
    for disease, by_sex in TARGET.items():
        for sex, target in by_sex.items():
            k = calibrate_factor(RATES, disease, sex, target)
            assert lifetime_risk(RATES, disease, sex, factor=k) == pytest.approx(target, abs=0.001)

In [11]:
# Cell 3 - RUN
print("Mean siblings born in 1960:", mean_siblings(1960))
print("Mean siblings born in 2022:", mean_siblings(2022))

Mean siblings born in 1960: 3.0999999999999996
Mean siblings born in 2022: 0.8999999999999999


Part A: FHRS

In [12]:
# Cell 2 - CODE
DOMAIN = {"CVD": ["CVD"], "T2D": ["T2D"], "CRD": ["asthma", "COPD"], "NEURO": ["depression", "GAD"]}

def onset_multiplier(disease, age, sex):
    if age is None: return 1.0
    if disease == "CVD":
        if age < 40: return 3.0
        if sex == "M": return 2.0 if age < 55 else 1.0 if age < 65 else 0.75
        return 2.0 if age < 65 else 1.0 if age < 75 else 0.75
    if disease in ("T2D", "COPD"):
        return 3.0 if age < 40 else 2.0 if age < 50 else 1.0 if age < 65 else 0.75
    if disease == "asthma":
        return 3.0 if age < 12 else 2.0 if age <= 25 else 1.0 if age <= 40 else 0.75
    if disease == "depression":
        return 3.0 if age < 18 else 2.0 if age <= 24 else 1.0 if age <= 40 else 0.75
    return 1.0

def fhrs(relatives, exposures, domain, on_date, cfg):
    R, S = cfg["fhrs"]["R"], cfg["fhrs"]["S"]
    bio, confirmed_first = 0.0, {}
    
    for r in relatives:
        for c in r["conditions"]:
            if c["disease"] in DOMAIN[domain] and c["year"] <= on_date.year:
                s = S["confirmed"] if c["confirmed"] else S["not_sure"]
                bio += R[r["degree"]] * onset_multiplier(c["disease"], c["age"], r["sex"]) * s
                if r["degree"] == 1 and c["confirmed"]:
                    confirmed_first[c["disease"]] = confirmed_first.get(c["disease"], 0) + 1
                break
                
    if any(n >= 2 for n in confirmed_first.values()):
        bio += cfg["fhrs"]["cluster_bonus"]
        
    E = 1.0 + sum(v for k, v in cfg["fhrs"]["penalties"][domain].items() if exposures.get(k))
    return round(bio * E, 2)

def band(score):
    if score == 0: return 0
    if score <= 1.75: return 1
    if score <= 3.50: return 2
    if score <= 6.00: return 3
    return 4

In [13]:
%%ipytest -q
# Cell 1 - TEST
from datetime import date

def rel(degree, sex, *conds):
    return {"degree": degree, "sex": sex, 
            "conditions": [{"disease": d, "year": y, "age": a, "confirmed": c} for d, y, a, c in conds]}

def test_cvd_onset_male_48_is_early():
    assert onset_multiplier("CVD", 48, "M") == 2.0

def test_cvd_onset_female_60_is_still_early():
    assert onset_multiplier("CVD", 60, "F") == 2.0

def test_unknown_onset_counts_as_typical():
    assert onset_multiplier("T2D", None, "F") == 1.0

def test_john_example_scores_4_17():
    relatives = [rel(1, "M", ("CVD", 1998, 48, True)), rel(1, "M", ("CVD", 2016, 38, True)), 
                 rel(2, "M", ("CVD", 2008, 60, False))]
    assert fhrs(relatives, {"secondhand_smoke": True}, "CVD", date(2026, 1, 1), CFG) == 4.17

def test_relative_diagnosed_after_visit_is_not_counted():
    assert fhrs([rel(1, "F", ("T2D", 2024, 64, True))], {}, "T2D", date(2023, 6, 1), CFG) == 0.0

def test_relative_diagnosed_before_visit_is_counted():
    assert fhrs([rel(1, "F", ("T2D", 2024, 64, True))], {}, "T2D", date(2025, 6, 1), CFG) == 0.5

def test_cluster_bonus_needs_two_confirmed_first_degree():
    r = [rel(1, "F", ("T2D", 2000, 55, True)), rel(1, "M", ("T2D", 2001, 55, False))]
    assert fhrs(r, {}, "T2D", date(2026, 1, 1), CFG) == 0.75

def test_crd_counts_asthma_and_copd_without_cluster_bonus():
    r = [rel(1, "F", ("asthma", 1990, 30, True)), rel(1, "M", ("COPD", 2010, 70, True))]
    assert fhrs(r, {}, "CRD", date(2026, 1, 1), CFG) == 0.88

def test_cluster_bonus_when_two_confirmed_have_same_disease():
    r = [rel(1, "F", ("asthma", 1990, 30, True)), rel(1, "M", ("asthma", 1995, 35, True))]
    assert fhrs(r, {}, "CRD", date(2026, 1, 1), CFG) == 2.0

def test_neuro_ignores_household_exposures():
    r = [rel(1, "F", ("depression", 2000, 30, True))]
    assert fhrs(r, {"secondhand_smoke": True}, "NEURO", date(2026, 1, 1), CFG) == 0.5

def test_band_edges_match_guide():
    assert [band(s) for s in [0, 0.05, 1.75, 1.76, 3.50, 3.51, 6.00, 6.01]] == [0, 1, 1, 2, 2, 3, 3, 4]

In [14]:
# Cell 3 - RUN
johns_relatives = [
    {"degree": 1, "sex": "M", "conditions": [{"disease": "CVD", "year": 1998, "age": 48, "confirmed": True}]},
    {"degree": 1, "sex": "M", "conditions": [{"disease": "CVD", "year": 2016, "age": 38, "confirmed": True}]},
    {"degree": 2, "sex": "M", "conditions": [{"disease": "CVD", "year": 2008, "age": 60, "confirmed": False}]}
]
score = fhrs(johns_relatives, {"secondhand_smoke": True}, "CVD", date(2026, 1, 1), CFG)
print(f"John's FHRS Score: {score}")
print(f"John's Band: {band(score)}")

John's FHRS Score: 4.17
John's Band: 3


Part B: Relatives' diseases

In [15]:
# Cell 2 - CODE
def simulate_family_diseases(relatives, cfg, rng, rate_fn, diseases, end_year=2026):
    first = min(r["birth_year"] for r in relatives)
    for year in range(first, end_year + 1):
        for r in relatives:
            age = year - r["birth_year"]
            if age < 0 or (r.get("death_year") and year > r["death_year"]):
                continue
            for d in diseases:
                if any(c["disease"] == d for c in r["conditions"]):
                    continue
                rate = rate_fn(d, age, r["sex"])
                
                if any(o is not r and o["degree"] == 1 and 
                       any(c["disease"] == d and c["year"] < year for c in o["conditions"]) 
                       for o in relatives):
                    rate *= cfg["family_multiplier"]
                    
                if rng.random() < min(rate, 0.99):
                    r["conditions"].append({"disease": d, "year": year, "age": age, "confirmed": True})

In [16]:
%%ipytest -q
# Cell 1 - TEST

def test_disease_clusters_in_families():
    rng = np.random.default_rng(7)
    flat = lambda disease, age, sex: 0.01 if age >= 30 else 0.0
    fams = []
    
    for _ in range(1500):
        fam = [{"degree": 1, "sex": "F", "birth_year": 1950, "conditions": []},
               {"degree": 1, "sex": "M", "birth_year": 1950, "conditions": []},
               {"degree": 1, "sex": "F", "birth_year": 1975, "conditions": []}]
        simulate_family_diseases(fam, CFG, rng, flat, diseases=["T2D"])
        fams.append(fam)
        
    has = lambda r: any(c["disease"] == "T2D" for c in r["conditions"])
    with_parent = [f for f in fams if has(f[0]) or has(f[1])]
    without = [f for f in fams if not (has(f[0]) or has(f[1]))]
    
    assert np.mean([has(f[2]) for f in with_parent]) > 1.3 * np.mean([has(f[2]) for f in without])

def test_dead_relative_gets_no_new_disease():
    fam = [{"degree": 1, "sex": "M", "birth_year": 1900, "death_year": 1950, "conditions": []}]
    simulate_family_diseases(fam, CFG, np.random.default_rng(1), lambda d, a, s: 0.5 if a >= 60 else 0.0, ["T2D"])
    assert all(c["year"] <= 1950 for c in fam[0]["conditions"])

In [17]:
# Cell 3 - RUN
rng = np.random.default_rng(7)
test_fam = [{"degree": 1, "sex": "F", "birth_year": 1960, "conditions": []},
            {"degree": 1, "sex": "M", "birth_year": 1962, "conditions": []}]
simulate_family_diseases(test_fam, CFG, rng, rate_fn, diseases=["T2D", "CVD"])
print("Simulated Family Conditions:", test_fam)

Simulated Family Conditions: [{'degree': 1, 'sex': 'F', 'birth_year': 1960, 'conditions': []}, {'degree': 1, 'sex': 'M', 'birth_year': 1962, 'conditions': []}]


Part C: Visit schedule

In [18]:
# Cell 2 - CODE
def add_months(d, months):
    m = d.month - 1 + int(round(months))
    return date(d.year + m // 12, m % 12 + 1, 1)

def schedule_visits(rng, cfg, tier_fn=None):
    tier_fn = tier_fn or (lambda d: "Low")
    d = date(cfg["window"]["start_year"] + int(rng.integers(0, 2)), int(rng.integers(1, 13)), 1)
    end = date(cfg["window"]["end_year"], 12, 31)
    dates = []
    
    while d <= end and len(dates) < cfg["visits"]["max"]:
        dates.append(d)
        gap = max(1, cfg["visit_gap_months"][tier_fn(d)] * rng.uniform(0.8, 1.2))
        d = add_months(d, gap)
    return dates

In [19]:
%%ipytest -q
# Cell 1 - TEST
from datetime import date

def test_visit_dates_go_forward_inside_window():
    dates = schedule_visits(np.random.default_rng(1), CFG)
    assert dates == sorted(dates)
    assert all(date(2021, 1, 1) <= d <= date(2026, 12, 31) for d in dates)
    assert 1 <= len(dates) <= 8

In [20]:
# Cell 3 - RUN
rng = np.random.default_rng(1)
visits = schedule_visits(rng, CFG)
print("Patient Visit Schedule:", visits)

Patient Visit Schedule: [datetime.date(2021, 7, 1), datetime.date(2022, 9, 1), datetime.date(2023, 7, 1), datetime.date(2024, 9, 1), datetime.date(2025, 8, 1), datetime.date(2026, 8, 1)]


Part D: Outcome labels

In [21]:
# Cell 2 - CODE
def label_for(diagnosis_date, index_date, years):
    if diagnosis_date is None:
        return 0
    if diagnosis_date <= index_date:
        return None
    end = date(index_date.year + years, index_date.month, index_date.day)
    return int(diagnosis_date <= end)

In [22]:
%%ipytest -q
# Cell 1 - TEST
from datetime import date

def test_label_is_1_when_diagnosed_inside_window():
    assert label_for(date(2024, 5, 1), date(2021, 3, 1), 5) == 1

def test_label_is_0_when_diagnosed_after_window():
    assert label_for(date(2027, 5, 1), date(2021, 3, 1), 5) == 0

def test_already_ill_at_index_is_excluded():
    assert label_for(date(2020, 1, 1), date(2021, 3, 1), 5) is None

In [23]:
# Cell 3 - RUN
index = date(2021, 1, 1)
dx = date(2025, 6, 1)
print(f"Diagnosed {dx}, Index {index} -> Label:", label_for(dx, index, 5))

Diagnosed 2025-06-01, Index 2021-01-01 -> Label: 1


Part E: Patients, values, and saving

In [ ]:
# Cell 2 - CODE  (Part E: hindi pa tapos, kailangan pa ng make_patient at make_visit_values)
rows_p, rows_r, rows_v, rows_t = [], [], [], []
ALL_DISEASES = ["CVD", "T2D", "asthma", "COPD", "depression", "GAD"]

# for pid in range(CFG["n_patients"]):
#     rng = np.random.default_rng([CFG["seed"], pid])
#     patient = make_patient(pid, rng)
#     simulate_family_diseases(patient["relatives"], CFG, rng, rate_fn, diseases=ALL_DISEASES)
#     for d in schedule_visits(rng, CFG):
#         v = make_visit_values(patient, d, rng)
#         v.update({f"fhrs_{dom}": fhrs(patient["relatives"], patient["exposures"], dom, d, CFG) for dom in DOMAIN})
#         rows_v.append({"patient_id": pid, "visit_date": d, **v})
#
# pd.DataFrame(rows_p).to_csv("output/patients.csv", index=False)
# pd.DataFrame(rows_r).to_csv("output/relatives.csv", index=False)
# pd.DataFrame(rows_v).to_csv("output/visits.csv", index=False)
# pd.DataFrame(rows_t).to_csv("output/training.csv", index=False)

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 14)

In [ ]:
%%ipytest -q
 Cell 1 - TEST
 The guide instructs writing these tests first (code not provided in sources):
1. test_every_value_stays_inside_its_dictionary_min_max_range
2. test_bmi_equals_weight_divided_by_height_squared_rounded_to_0_1
3. test_a_progressors_hba1c_is_higher_at_the_last_visit_than_at_the_first
4. test_gestational_diabetes_appears_only_for_female_patients
5. test_pack_years_never_go_down_between_visits

In [ ]:
# Cell 3 - RUN
# (Cannot run full generator loop until make_patient and make_visit_values are implemented)
print("Part E loop sketch created.")

Part E loop sketch created.
